# Research notebook archive

Original project notebook with outputs and personal metadata removed. Existing Colab paths and experiment settings require configuration; see the repository README before running.

# Stage 3 Dataset Construction Utilities

Utility notebook for building:
- train/validation/test splits
- non-overlapping demographic partitions
- natural/balanced/weighted datasets
- evaluation datasets
- metadata and leakage checks

In [ ]:

import os
import json
import string
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:

# ============================================================
# BASIC HELPERS
# ============================================================

def ensure_row_ids(df):
    df = df.reset_index(drop=True).copy()

    if "row_id" not in df.columns:
        df["row_id"] = df.index

    return df


def create_directories(base_save_dir):

    dataset_dir = Path(base_save_dir)

    for subdir in [
        "splits",
        "partitions",
        "natural",
        "balanced",
        "weighted",
        "evaluation/validation",
        "evaluation/test",
        "metadata",
    ]:
        (dataset_dir / subdir).mkdir(parents=True, exist_ok=True)

    return dataset_dir


def create_master_splits(
    df,
    train_frac=0.70,
    val_frac=0.15,
    test_frac=0.15,
    random_seed=42,
):

    train_pool_df, temp_df = train_test_split(
        df,
        test_size=(1 - train_frac),
        random_state=random_seed,
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=test_frac / (val_frac + test_frac),
        random_state=random_seed,
    )

    return (
        train_pool_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


In [ ]:

# ============================================================
# SAMPLING FUNCTIONS
# ============================================================

def sample_balanced(
    df,
    target_col,
    n_per_class,
    seed=42,
    use_weights=True,
):

    pieces = []

    for class_value, group in df.dropna(
        subset=[target_col]
    ).groupby(
        target_col,
        observed=True,
    ):

        replace = len(group) < n_per_class

        # ----------------------------------------
        # WITHIN-CLASS ACS WEIGHTING
        # ----------------------------------------

        if use_weights and "PERWT" in group.columns:

            weights = (
                group["PERWT"]
                / group["PERWT"].sum()
            )

        else:
            weights = None

        sampled = group.sample(
            n=n_per_class,
            replace=replace,
            weights=weights,
            random_state=seed,
        )

        pieces.append(sampled)

    out = pd.concat(pieces, axis=0)

    out = out.sample(
        frac=1,
        random_state=seed,
    ).reset_index(drop=True)

    return out

def sample_natural(
    df,
    target_col,
    n_total,
    seed=42,
    use_weights=True,
):

    clean = df.dropna(
        subset=[target_col]
    ).copy()

    replace = len(clean) < n_total

    # ----------------------------------------
    # ACS WEIGHTED SAMPLING
    # ----------------------------------------

    if use_weights and "PERWT" in clean.columns:

        weights = (
            clean["PERWT"]
            / clean["PERWT"].sum()
        )

    else:
        weights = None

    out = clean.sample(
        n=n_total,
        replace=replace,
        weights=weights,
        random_state=seed,
    )

    return out.reset_index(drop=True)


def make_class_weights(
    df,
    target_col,
):

    counts = (
        df[target_col]
        .value_counts()
    )

    weights = 1 / counts

    # normalize around 1
    weights = (
        weights
        / weights.mean()
    )

    return weights.to_dict()


In [ ]:
# ============================================================
# CREATE RANDOM NON-OVERLAPPING PARTITIONS
# ============================================================

def create_target_partitions(
    train_pool_df,
    targets,
    train_per_class=1000,
    random_seed=42,
):

    rng = np.random.default_rng(random_seed)

    available_df = train_pool_df.sample(
        frac=1,
        random_state=random_seed,
    ).reset_index(drop=True)

    partition_dfs = {}
    partition_metadata = []

    used_row_ids = set()

    for target_col in targets:

        # ----------------------------------------
        # HOW MANY ROWS NEEDED
        # ----------------------------------------

        n_classes = (
            available_df[target_col]
            .dropna()
            .nunique()
        )

        n_needed = (
            n_classes * train_per_class
        )

        # ----------------------------------------
        # AVAILABLE UNUSED ROWS
        # ----------------------------------------

        candidate_df = available_df[
            ~available_df["row_id"].isin(
                used_row_ids
            )
        ].copy()

        candidate_df = candidate_df.dropna(
            subset=[target_col]
        )

        # ----------------------------------------
        # RANDOM PARTITION
        # ----------------------------------------

        partition = candidate_df.sample(
            n=n_needed,
            replace=False,
            random_state=random_seed,
        )

        used_row_ids.update(
            partition["row_id"].tolist()
        )

        partition_dfs[target_col] = partition

        # ----------------------------------------
        # METADATA
        # ----------------------------------------

        partition_metadata.append({
            "target": target_col,
            "rows": len(partition),
            "unique_rows":
                partition["row_id"].nunique(),
        })

        print("\n")
        print("=" * 80)
        print(target_col)
        print("=" * 80)

        print(
            partition[target_col]
            .value_counts(normalize=True)
        )

    return (
        partition_dfs,
        pd.DataFrame(partition_metadata),
    )

In [ ]:
# ============================================================
# BUILD METHOD DATASETS
# ============================================================

def build_method_datasets(
    partition_dfs,
    targets,
    dataset_dir,
    train_per_class=1000,
    random_seed=42,
):

    outputs = {}

    for target_col in targets:

        print("\n")
        print("=" * 80)
        print(f"BUILDING DATASETS FOR {target_col}")
        print("=" * 80)

        partition = partition_dfs[target_col]

        n_total = len(partition)

        # ====================================================
        # NATURAL
        # ====================================================

        natural_sample = sample_natural(
            df=partition,
            target_col=target_col,
            n_total=n_total,
            seed=random_seed,
        )

        # ====================================================
        # BALANCED
        # ====================================================

        balanced_sample = sample_balanced(
            df=partition,
            target_col=target_col,
            n_per_class=train_per_class,
            seed=random_seed,
        )

        # ====================================================
        # WEIGHTED
        # ====================================================

        weighted_sample = natural_sample.copy()

        class_weights = make_class_weights(
            weighted_sample,
            target_col=target_col,
        )

        weighted_sample["class_weight"] = (
            weighted_sample[target_col]
            .map(class_weights)
        )

        # ====================================================
        # STORE
        # ====================================================

        method_outputs = {
            "natural": natural_sample,
            "balanced": balanced_sample,
            "weighted": weighted_sample,
        }

        outputs[target_col] = method_outputs

        # ====================================================
        # SAVE
        # ====================================================

        for method_name, dataset_df in method_outputs.items():

            save_path = (
                dataset_dir
                / method_name
                / f"{target_col}.parquet"
            )

            dataset_df.to_parquet(save_path)

            print(f"Saved → {save_path}")

    return outputs

In [ ]:
# ============================================================
# BUILD + SAVE EVALUATION DATASETS
# ============================================================

def build_eval_datasets(
    eval_df,
    targets,
    dataset_dir,
    eval_per_class=300,
    dataset_name="validation",
    random_seed=42,
):

    outputs = {}

    # ----------------------------
    # SAVE DIRECTORY
    # ----------------------------

    save_dir = (
    dataset_dir
    / dataset_name
    / "balanced"
)

    save_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # ----------------------------
    # BUILD DATASETS
    # ----------------------------

    for target_col in targets:

        sampled = sample_balanced(
            df=eval_df,
            target_col=target_col,
            n_per_class=eval_per_class,
            seed=random_seed,
        )

        outputs[target_col] = sampled

        # ------------------------
        # SAVE
        # ------------------------

        save_path = (
            save_dir
            / f"{target_col}.parquet"
        )

        sampled.to_parquet(save_path)

        print(f"Saved → {save_path}")

    return outputs

In [ ]:

# ============================================================
# CONFIG
# ============================================================

RANDOM_SEED = 42

TARGETS = [
    "SEX",
    'HEALTH_INSURANCE',
    'ENGLISH_STATUS',
    "AGE_BINS",
    "EDUCATION_LEVEL",
    'INCTOT_BINS',
    "RACE_ETHNICITY",
]

TRAIN_PER_CLASS = 1500
EVAL_PER_CLASS = 500

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15


In [ ]:
# ============================================================
# SETUP
# ============================================================
df = pd.read_parquet("/content/drive/MyDrive/NLP_project/data/demographic_data_clean.parquet")
df = ensure_row_ids(df)
BASE_SAVE_DIR = "/content/drive/MyDrive/NLP_project/data/stage3_outputs"
DATASET_DIR = create_directories(
    f"{BASE_SAVE_DIR}/stage3_training_datasets"
)


# ============================================================
# REMOVE UNUSED CATEGORIES
# ============================================================

for col in TARGETS:

    if pd.api.types.is_categorical_dtype(df[col]):

        df[col] = (
            df[col]
            .cat.remove_unused_categories()
        )

        print("\n")
        print("=" * 80)
        print(col)
        print("=" * 80)

        print(df[col].value_counts())
# ============================================================
# CREATE TRAIN / VAL / TEST SPLITS
# ============================================================

train_pool_df, val_df, test_df = (
    create_master_splits(
        df=df,
        train_frac=TRAIN_FRAC,
        val_frac=VAL_FRAC,
        test_frac=TEST_FRAC,
        random_seed=RANDOM_SEED,
    )
)

print("\n")
print("=" * 80)
print("MASTER SPLITS")
print("=" * 80)

print(f"Train pool: {len(train_pool_df):,}")
print(f"Validation: {len(val_df):,}")
print(f"Test:       {len(test_df):,}")

# ============================================================
# CREATE NON-OVERLAPPING TARGET PARTITIONS
# ============================================================

partition_dfs, partition_metadata = (
    create_target_partitions(
        train_pool_df=train_pool_df,
        targets=TARGETS,
        train_per_class=TRAIN_PER_CLASS,
        random_seed=RANDOM_SEED,
    )
)

print("\n")
print("=" * 80)
print("PARTITION METADATA")
print("=" * 80)

display(partition_metadata)

# ============================================================
# SAVE PARTITIONS
# ============================================================

for target_col, partition_df in partition_dfs.items():

    save_path = (
        DATASET_DIR
        / "partitions"
        / f"{target_col}.parquet"
    )

    partition_df.to_parquet(save_path)

    print(f"Saved partition → {save_path}")



In [ ]:
# ============================================================
# PRETTY SUMMARY
# ============================================================

for col in TARGETS:
    counts = (
        df[col]
        .value_counts(dropna=False)
        .sort_index()
    )

    proportions = (
        df[col]
        .value_counts(
            normalize=True,
            dropna=False,
        )
        .sort_index()
    )

    print("\n")
    print("=" * 80)
    print(col)
    print("=" * 80)

    for category in counts.index:

        print(
            f"{category:<20} | "
            f"Count: {counts[category]:>8,} | "
            f"Prop: {proportions[category]:.4f}"
        )

In [ ]:
# ============================================================
# BUILD TRAINING DATASETS
# ============================================================

train_datasets = build_method_datasets(
    partition_dfs=partition_dfs,
    targets=TARGETS,
    dataset_dir=DATASET_DIR,
    train_per_class=TRAIN_PER_CLASS,
    random_seed=RANDOM_SEED,
)

# ============================================================
# VALIDATION DATASETS
# ============================================================

validation_datasets = build_eval_datasets(
    eval_df=val_df,
    targets=TARGETS,
    dataset_dir=DATASET_DIR,
    eval_per_class=EVAL_PER_CLASS,
    dataset_name="validation",
    random_seed=RANDOM_SEED,
)

# ============================================================
# TEST DATASETS
# ============================================================

test_datasets = build_eval_datasets(
    eval_df=test_df,
    targets=TARGETS,
    dataset_dir=DATASET_DIR,
    eval_per_class=EVAL_PER_CLASS,
    dataset_name="test",
    random_seed=RANDOM_SEED,
)

In [ ]:
# ============================================================
# SAVE RAW SPLITS
# ============================================================

train_pool_df.to_parquet(
    DATASET_DIR / "splits" / "train_pool.parquet"
)

val_df.to_parquet(
    DATASET_DIR / "splits" / "validation_raw.parquet"
)

test_df.to_parquet(
    DATASET_DIR / "splits" / "test_raw.parquet"
)

print("\nSaved raw splits.")

# ============================================================
# SAVE METADATA
# ============================================================

partition_metadata.to_csv(
    DATASET_DIR
    / "metadata"
    / "partition_metadata.csv",
    index=False,
)

print("\nSaved metadata.")

# ============================================================
# LEAKAGE CHECKS
# ============================================================

print("\n")
print("=" * 80)
print("LEAKAGE CHECKS")
print("=" * 80)

train_ids = set(train_pool_df["row_id"])
val_ids = set(val_df["row_id"])
test_ids = set(test_df["row_id"])

print("Train ∩ Validation:", len(train_ids & val_ids))
print("Train ∩ Test:      ", len(train_ids & test_ids))
print("Validation ∩ Test: ", len(val_ids & test_ids))

print("\n")
print("=" * 80)
print("PARTITION OVERLAP CHECKS")
print("=" * 80)

for i, t1 in enumerate(TARGETS):

    for t2 in TARGETS[i + 1:]:

        ids1 = set(
            partition_dfs[t1]["row_id"]
        )

        ids2 = set(
            partition_dfs[t2]["row_id"]
        )

        overlap = len(ids1 & ids2)

        print(f"{t1} ∩ {t2}: {overlap}")

print("\n")
print("=" * 80)
print("DONE")
print("=" * 80)

print(DATASET_DIR)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ============================================================
# ACS DISTRIBUTION HELPER
# ============================================================

def get_weighted_distribution(
    df,
    target_col,
):

    weighted_dist = (
        df.groupby(
            target_col,
            observed=True,
        )["PERWT"]
        .sum()
    )

    weighted_dist = (
        weighted_dist
        / weighted_dist.sum()
    )

    return weighted_dist.sort_index()

# ============================================================
# PLOT BY TRAINING METHOD
# ============================================================

METHODS = [
    "natural",
    "balanced",
    "weighted",
]

for method in METHODS:

    print("\n")
    print("=" * 80)
    print(method.upper())
    print("=" * 80)

    n_targets = len(TARGETS)

    fig, axes = plt.subplots(
        n_targets,
        1,
        figsize=(12, 5 * n_targets),
    )

    if n_targets == 1:
        axes = [axes]

    # ========================================================
    # LOOP THROUGH TARGETS
    # ========================================================

    for ax, target_col in zip(axes, TARGETS):

        # ----------------------------------------------------
        # ACS DISTRIBUTION
        # ----------------------------------------------------

        acs_dist = get_weighted_distribution(
            df,
            target_col,
        )

        # ----------------------------------------------------
        # TRAIN DISTRIBUTION
        # ----------------------------------------------------

        dataset_df = (
            train_datasets[target_col][method]
        )

        train_dist = (
            dataset_df[target_col]
            .value_counts(normalize=True)
            .sort_index()
        )

        train_dist = train_dist.reindex(
            acs_dist.index,
            fill_value=0,
        )

        # ----------------------------------------------------
        # PLOT
        # ----------------------------------------------------

        x = np.arange(len(acs_dist.index))

        width = 0.35

        ax.bar(
            x - width / 2,
            acs_dist.values,
            width=width,
            label="ACS Weighted",
        )

        ax.bar(
            x + width / 2,
            train_dist.values,
            width=width,
            label=method,
        )

        # ----------------------------------------------------
        # FORMATTING
        # ----------------------------------------------------

        ax.set_xticks(x)

        ax.set_xticklabels(
            acs_dist.index,
            rotation=30,
            ha="right",
        )

        ax.set_ylabel("Proportion")

        ax.set_title(
            f"{target_col}"
        )

        ax.legend()

    # ========================================================
    # FIGURE TITLE
    # ========================================================

    fig.suptitle(
        f"{method.upper()} Dataset Distributions",
        fontsize=18,
        y=1.02,
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# CHECK RAW + BALANCED VALIDATION/TEST DISTRIBUTIONS
# ============================================================

for target_col in TARGETS:

    print("\n")
    print("=" * 80)
    print(target_col)
    print("=" * 80)

    # --------------------------------------------------------
    # ACS WEIGHTED DISTRIBUTION
    # --------------------------------------------------------

    acs_dist = (
        df.groupby(
            target_col,
            observed=True,
        )["PERWT"]
        .sum()
    )

    acs_dist = (
        acs_dist
        / acs_dist.sum()
    ).sort_index()

    # ========================================================
    # RAW VALIDATION
    # ========================================================

    raw_val_counts = (
        val_df[target_col]
        .value_counts()
        .sort_index()
    )

    raw_val_props = (
        val_df[target_col]
        .value_counts(normalize=True)
        .sort_index()
    )

    # ========================================================
    # BALANCED VALIDATION
    # ========================================================

    balanced_val_counts = (
        validation_datasets[target_col][target_col]
        .value_counts()
        .sort_index()
    )

    balanced_val_props = (
        validation_datasets[target_col][target_col]
        .value_counts(normalize=True)
        .sort_index()
    )

    # ========================================================
    # RAW TEST
    # ========================================================

    raw_test_counts = (
        test_df[target_col]
        .value_counts()
        .sort_index()
    )

    raw_test_props = (
        test_df[target_col]
        .value_counts(normalize=True)
        .sort_index()
    )

    # ========================================================
    # BALANCED TEST
    # ========================================================

    balanced_test_counts = (
        test_datasets[target_col][target_col]
        .value_counts()
        .sort_index()
    )

    balanced_test_props = (
        test_datasets[target_col][target_col]
        .value_counts(normalize=True)
        .sort_index()
    )

    # ========================================================
    # COMBINE
    # ========================================================

    summary_df = pd.DataFrame({

        # ----------------------------------------------------
        # ACS
        # ----------------------------------------------------

        "ACS_weighted":
            acs_dist,

        # ----------------------------------------------------
        # RAW VALIDATION
        # ----------------------------------------------------

        "raw_val_count":
            raw_val_counts,

        "raw_val_prop":
            raw_val_props,

        # ----------------------------------------------------
        # BALANCED VALIDATION
        # ----------------------------------------------------

        "balanced_val_count":
            balanced_val_counts,

        "balanced_val_prop":
            balanced_val_props,

        # ----------------------------------------------------
        # RAW TEST
        # ----------------------------------------------------

        "raw_test_count":
            raw_test_counts,

        "raw_test_prop":
            raw_test_props,

        # ----------------------------------------------------
        # BALANCED TEST
        # ----------------------------------------------------

        "balanced_test_count":
            balanced_test_counts,

        "balanced_test_prop":
            balanced_test_props,
    })

    summary_df = summary_df.fillna(0)

    display(summary_df)

In [ ]:
# ============================================================
# SAVE RAW VALIDATION DATASETS
# ============================================================

raw_validation_dir = (
    DATASET_DIR
    / "validation"
    / "raw"
)

raw_validation_dir.mkdir(
    parents=True,
    exist_ok=True,
)

for target_col in TARGETS:

    save_df = val_df.dropna(
        subset=[target_col]
    ).copy()

    save_path = (
        raw_validation_dir
        / f"{target_col}.parquet"
    )

    save_df.to_parquet(save_path)

    print(f"Saved → {save_path}")

In [ ]:
# ============================================================
# SAVE RAW TEST DATASETS
# ============================================================

raw_test_dir = (
    DATASET_DIR
    / "test"
    / "raw"
)

raw_test_dir.mkdir(
    parents=True,
    exist_ok=True,
)

for target_col in TARGETS:

    save_df = test_df.dropna(
        subset=[target_col]
    ).copy()

    save_path = (
        raw_test_dir
        / f"{target_col}.parquet"
    )

    save_df.to_parquet(save_path)

    print(f"Saved → {save_path}")

Covariate Balance Check

In [ ]:
from scipy.stats import ttest_ind, chi2_contingency

# ============================================================
# SETTINGS
# ============================================================

NUMERIC_COLS = [
    "HHINCOME",
    "NCHILD",
    "UHRSWORK",
    "INCWAGE",
]

CATEGORICAL_COLS = [
    "SEX",
    "AGE_BINS",
    "RACE_ETHNICITY",
    "EDUCATION_LEVEL",
    "EMPLOYMENT_STATUS",
    "ENGLISH_STATUS",
    "HEALTH_INSURANCE",
    "MARITAL_STATUS",
]

ALPHA = 0.01

# ============================================================
# BASELINE BALANCE TESTS
# ============================================================

results = []

for target_col in TARGETS:

    classes = sorted(
        train_datasets[target_col]["natural"][
            target_col
        ].dropna().unique()
    )

    for class_value in classes:

        # ----------------------------------------------------
        # SUBSETS
        # ----------------------------------------------------

        natural_df = (
            train_datasets[target_col]["natural"]
        )

        balanced_df = (
            train_datasets[target_col]["balanced"]
        )

        weighted_df = (
            train_datasets[target_col]["weighted"]
        )

        natural_subset = natural_df[
            natural_df[target_col]
            == class_value
        ]

        balanced_subset = balanced_df[
            balanced_df[target_col]
            == class_value
        ]

        weighted_subset = weighted_df[
            weighted_df[target_col]
            == class_value
        ]

        # ====================================================
        # NUMERIC VARIABLES
        # ====================================================

        for col in NUMERIC_COLS:

            if col not in natural_subset.columns:
                continue

            # --------------------------------------------
            # NATURAL vs BALANCED
            # --------------------------------------------

            stat, p = ttest_ind(
                natural_subset[col].dropna(),
                balanced_subset[col].dropna(),
                equal_var=False,
            )

            if p < ALPHA:

                results.append({
                    "target": target_col,
                    "class": class_value,
                    "comparison":
                        "natural_vs_balanced",
                    "variable": col,
                    "test": "t-test",
                    "p_value": p,
                    "natural_mean":
                        natural_subset[col].mean(),
                    "other_mean":
                        balanced_subset[col].mean(),
                })

            # --------------------------------------------
            # NATURAL vs WEIGHTED
            # --------------------------------------------

            stat, p = ttest_ind(
                natural_subset[col].dropna(),
                weighted_subset[col].dropna(),
                equal_var=False,
            )

            if p < ALPHA:

                results.append({
                    "target": target_col,
                    "class": class_value,
                    "comparison":
                        "natural_vs_weighted",
                    "variable": col,
                    "test": "t-test",
                    "p_value": p,
                    "natural_mean":
                        natural_subset[col].mean(),
                    "other_mean":
                        weighted_subset[col].mean(),
                })

        # ====================================================
        # CATEGORICAL VARIABLES
        # ====================================================

        for col in CATEGORICAL_COLS:

            if col == target_col:
                continue

            # --------------------------------------------
            # NATURAL vs BALANCED
            # --------------------------------------------

            contingency = pd.concat([
                natural_subset[col].value_counts(),
                balanced_subset[col].value_counts(),
            ],
            axis=1).fillna(0)

            contingency.columns = [
                "natural",
                "balanced",
            ]

            chi2, p, _, _ = chi2_contingency(
                contingency
            )

            if p < ALPHA:

                results.append({
                    "target": target_col,
                    "class": class_value,
                    "comparison":
                        "natural_vs_balanced",
                    "variable": col,
                    "test": "chi-square",
                    "p_value": p,
                })

            # --------------------------------------------
            # NATURAL vs WEIGHTED
            # --------------------------------------------

            contingency = pd.concat([
                natural_subset[col].value_counts(),
                weighted_subset[col].value_counts(),
            ],
            axis=1).fillna(0)

            contingency.columns = [
                "natural",
                "weighted",
            ]

            chi2, p, _, _ = chi2_contingency(
                contingency
            )

            if p < ALPHA:

                results.append({
                    "target": target_col,
                    "class": class_value,
                    "comparison":
                        "natural_vs_weighted",
                    "variable": col,
                    "test": "chi-square",
                    "p_value": p,
                })

# ============================================================
# RESULTS
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "p_value"
)

print("\n")
print("=" * 80)
print("SIGNIFICANT BASELINE DIFFERENCES")
print("=" * 80)

display(results_df)

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd
import numpy as np

# ============================================================
# SETTINGS
# ============================================================

NUMERIC_COLS = [
    "HHINCOME",
    "NCHILD",
    "UHRSWORK",
    "INCWAGE",
]

CATEGORICAL_COLS = [
    "SEX",
    "AGE_BINS",
    "RACE_ETHNICITY",
    "EDUCATION_LEVEL",
    "EMPLOYMENT_STATUS",
    "ENGLISH_STATUS",
    "HEALTH_INSURANCE",
    "MARITAL_STATUS",
]

METHOD_COMPARISONS = [
    ("natural", "balanced"),
    ("natural", "weighted"),
]

# ============================================================
# EFFECT SIZE HELPERS
# ============================================================

def cohens_d(x1, x2):

    x1 = x1.dropna()
    x2 = x2.dropna()

    n1 = len(x1)
    n2 = len(x2)

    if n1 < 2 or n2 < 2:
        return np.nan

    s1 = x1.std(ddof=1)
    s2 = x2.std(ddof=1)

    pooled_sd = np.sqrt(
        (
            ((n1 - 1) * s1**2)
            + ((n2 - 1) * s2**2)
        )
        / (n1 + n2 - 2)
    )

    if pooled_sd == 0:
        return 0

    d = (
        x1.mean()
        - x2.mean()
    ) / pooled_sd

    return d


def cramers_v(contingency_table):

    chi2, _, _, _ = chi2_contingency(
        contingency_table
    )

    n = contingency_table.sum().sum()

    if n == 0:
        return np.nan

    phi2 = chi2 / n

    r, k = contingency_table.shape

    return np.sqrt(
        phi2 / min(k - 1, r - 1)
    )

# ============================================================
# EFFECT SIZE ANALYSIS
# ============================================================

results = []

for target_col in TARGETS:

    classes = sorted(
        train_datasets[target_col]["natural"][
            target_col
        ].dropna().unique()
    )

    for class_value in classes:

        for method_a, method_b in METHOD_COMPARISONS:

            df_a = (
                train_datasets[target_col][method_a]
            )

            df_b = (
                train_datasets[target_col][method_b]
            )

            subset_a = df_a[
                df_a[target_col]
                == class_value
            ]

            subset_b = df_b[
                df_b[target_col]
                == class_value
            ]

            # =================================================
            # NUMERIC VARIABLES
            # =================================================

            for col in NUMERIC_COLS:

                if col not in subset_a.columns:
                    continue

                d = cohens_d(
                    subset_a[col],
                    subset_b[col],
                )

                results.append({
                    "target": target_col,
                    "class": class_value,
                    "comparison":
                        f"{method_a}_vs_{method_b}",
                    "variable": col,
                    "variable_type": "numeric",
                    "effect_size": d,
                    "abs_effect_size": abs(d),
                    "mean_a":
                        subset_a[col].mean(),
                    "mean_b":
                        subset_b[col].mean(),
                })

            # =================================================
            # CATEGORICAL VARIABLES
            # =================================================

            for col in CATEGORICAL_COLS:

                if col == target_col:
                    continue

                contingency = pd.concat([
                    subset_a[col].value_counts(),
                    subset_b[col].value_counts(),
                ],
                axis=1).fillna(0)

                contingency.columns = [
                    method_a,
                    method_b,
                ]

                v = cramers_v(
                    contingency
                )

                results.append({
                    "target": target_col,
                    "class": class_value,
                    "comparison":
                        f"{method_a}_vs_{method_b}",
                    "variable": col,
                    "variable_type":
                        "categorical",
                    "effect_size": v,
                    "abs_effect_size": abs(v),
                })

# ============================================================
# RESULTS DATAFRAME
# ============================================================

effect_df = pd.DataFrame(results)

effect_df = effect_df.sort_values(
    "abs_effect_size",
    ascending=False,
)

# ============================================================
# DISPLAY LARGEST EFFECTS
# ============================================================

print("\n")
print("=" * 80)
print("LARGEST EFFECT SIZES")
print("=" * 80)

display(
    effect_df.head(50)
)

# ============================================================
# SUMMARY BY COMPARISON
# ============================================================

summary = (
    effect_df
    .groupby(
        [
            "comparison",
            "variable_type",
        ]
    )["abs_effect_size"]
    .agg([
        "mean",
        "median",
        "max",
    ])
)

print("\n")
print("=" * 80)
print("EFFECT SIZE SUMMARY")
print("=" * 80)

display(summary)